# Validação do monitor ao vivo: `gold_orders_live` recalculada por outro caminho (etapa 5.4)

Confere se `olist.gold.gold_orders_live` e a lista de pedidos em risco reproduzem o que se calcula **direto da silver e da gold batch**, sem usar nenhuma função do notebook `01_gold_orders_live` para montar o número esperado. Cada verificação grava uma linha em `olist.quality.validacao_live` (esperado × obtido × ok) e o notebook **falha no final** se alguma estiver reprovada.

**Antes de rodar:** a bronze completa precisa estar ingerida e os pipelines `olist.silver` e `olist.gold` atualizados com ela, e `01_gold_orders_live` executado duas vezes: com `teste_leitura = nao` (a execução real) e depois com `teste_leitura = conferencia`, para que `fact_orders`, a silver e o estado ao vivo falem dos mesmos eventos e a idempotência esteja medida.

| # | Grupo | Verificações | Esperado |
|---|---|---|---|
| 1 | Estado | `linhas_igual_pedidos_com_evento`, `granularidade_sem_duplicata`, `status_igual_silver`, `status_igual_fact_orders`, `entrega_igual_fact_orders`, `etapas_igual_silver`, `estimada_igual_silver`, `pedido_sem_evento_fora_do_live`, `evento_sem_pedido` | uma linha por pedido com evento; status, sequência, instantes e data estimada iguais ao recalculado da silver e à gold batch (critérios 1 a 3 do desenho) |
| 2 | Micro-lotes | `nenhum_evento_perdido`, `latencia_dentro_do_sla`, `latencia_maxima_s`, `tardios_contados` | cada evento distinto chegou uma vez ao `MERGE`; nenhum micro-lote incremental passou dos 15 min do SLA; tardios contados (critérios 5 e 7) |
| 3 | Risco | `data_simulada_igual`, `risco_igual_calculo_independente`, `risco_so_pedidos_abertos`, `lista_*` | o nível de risco de cada pedido igual a uma regra reescrita em SQL; pedido fechado sem risco; a lista é a view com atrasados e em risco (critério 6) |
| 4 | Idempotência | `conferencia_*`, `replay_checkpoint_novo`, `idempotencia_sem_arquivo_novo` | a conferência do monitor é recente e tem o mesmo hash do estado atual; reler tudo com checkpoint novo e rodar sem arquivo novo dão o mesmo estado (critério 4) |

## Configuração

Constantes, o identificador desta validação e a lista onde as verificações se acumulam.

In [ ]:
# O QUE FAZ: define as constantes (catálogo, tabela de resultados, SLA) e cria a lista
#   onde as verificações vão se acumulando. Nenhum dado é lido aqui.
#
# RECURSOS:
# - uuid.uuid4(): identificador aleatório único desta validação (agrupa as linhas dela).
# - f"...": f-string, insere variáveis no texto.
import uuid

from pyspark.sql import functions as F

CATALOG = "olist"
TABELA_RESULTADOS = f"{CATALOG}.quality.validacao_live"
LIVE = f"{CATALOG}.gold.gold_orders_live"
BATCHES = f"{CATALOG}.ops.streaming_batches"
VIEW_RISCO = f"{CATALOG}.gold.vw_pedidos_em_risco"
EVENTOS_BRONZE = f"{CATALOG}.bronze.order_status_events"
SLA_SEGUNDOS = 15 * 60       # ADR 0012
LOTE_GRANDE = 10000          # micro-lotes com mais eventos que isso são carga inicial ou recuperação, não ciclo
N_EM_RISCO = 2               # ADR 0011
ABERTOS = ("created", "approved", "invoiced", "processing", "shipped")
VALIDACAO_ID = str(uuid.uuid4())

resultados = []  # (entidade, verificacao, esperado, obtido, escala, ok, detalhe)
print(f"validacao_id: {VALIDACAO_ID}")

## Tabela de resultados e funções auxiliares

`registrar` guarda uma contagem (esperado e obtido inteiros; sem esperado, só registro). `n` roda uma consulta que devolve um número só. `silver` e `gold` montam nomes de tabela.

In [ ]:
# O QUE FAZ: cria a tabela de resultados (se não existir) e declara as funções auxiliares.
#
# FUNÇÕES DECLARADAS:
# - gold(nome) / silver(nome): nome completo da tabela (olist.gold.x, olist.silver.x).
# - n(consulta): executa um SQL e devolve o primeiro valor da primeira linha como número Python (nulo vira 0).
# - registrar(entidade, verificacao, esperado, obtido, detalhe): ok = (esperado == obtido); sem esperado
#   (None), ok fica nulo = "só registro".
#
# FUNÇÕES NATIVAS:
# - spark.sql(texto): executa SQL e devolve um DataFrame. .first()[0]: primeira coluna da primeira linha.
def gold(nome):
    return f"{CATALOG}.gold.{nome}"


def silver(nome):
    return f"{CATALOG}.silver.{nome}"


def n(consulta):
    valor = spark.sql(consulta).first()[0]
    return 0 if valor is None else int(valor)


def registrar(entidade, verificacao, esperado, obtido, detalhe=""):
    ok = None if esperado is None else (esperado == obtido)
    resultados.append((entidade, verificacao, esperado, obtido, 1, ok, detalhe))
    marca = "  " if ok is None else ("OK" if ok else "XX")
    print(f"[{marca}] {entidade:22} {verificacao:36} esperado={esperado} obtido={obtido} {detalhe}")


spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABELA_RESULTADOS} (
  validacao_id  STRING    COMMENT 'Identificador da execução da validação',
  verificado_em TIMESTAMP COMMENT 'Momento da verificação',
  entidade      STRING    COMMENT 'Tabela ou objeto verificado',
  verificacao   STRING    COMMENT 'Nome da verificação',
  esperado      BIGINT    COMMENT 'Valor esperado (recalculado da silver, da gold batch ou das regras)',
  obtido        BIGINT    COMMENT 'Valor obtido em gold_orders_live, na view ou em ops.streaming_batches',
  escala        BIGINT    COMMENT 'Multiplicador dos valores (1 = contagem)',
  ok            BOOLEAN   COMMENT 'Verdadeiro se esperado = obtido; nulo se a linha é só registro',
  detalhe       STRING    COMMENT 'Texto explicativo'
)
COMMENT 'Resultado da validação do monitor ao vivo: gold_orders_live, micro-lotes, risco e idempotência recalculados por outro caminho (etapa 5.4).'
''')

## Grupo 1 — estado do pedido

O esperado vem da **silver** (`order_status_events`, `orders`) e da **gold batch** (`fact_orders`), nunca de `gold_orders_live`. Para o status, `max_by(status, event_sequence)` é a forma independente de pegar "o evento de maior sequência" (o notebook do monitor usa `row_number` e `MERGE`). As verificações de diferença usam `FULL OUTER JOIN`: um pedido só de um lado conta como divergência.

In [ ]:
# O QUE FAZ: registra as verificações do grupo 1 (estado do pedido).
#
# FUNÇÕES NATIVAS (SQL):
# - count(DISTINCT x): valores diferentes. count_if(cond): linhas que cumprem a condição.
# - max_by(valor, ordem): valor da linha em que "ordem" é a maior. min/max com CASE WHEN: o instante de um status.
# - FULL OUTER JOIN: mantém os pedidos que existem só de um lado. LEFT ANTI JOIN: linhas sem par.
# - a <=> b: igualdade que trata nulo = nulo como verdadeiro.
ESPERADO = f'''
  SELECT order_id, max_by(status, event_sequence) AS st, max(event_sequence) AS seq,
         max(CASE WHEN status = 'created'   THEN event_ts END) AS created_at,
         max(CASE WHEN status = 'approved'  THEN event_ts END) AS approved_at,
         max(CASE WHEN status = 'shipped'   THEN event_ts END) AS shipped_at,
         max(CASE WHEN status = 'delivered' THEN event_ts END) AS delivered_at
  FROM {silver('order_status_events')} GROUP BY order_id
'''

registrar("gold_orders_live", "linhas_igual_pedidos_com_evento",
          n(f"SELECT count(DISTINCT order_id) FROM {silver('order_status_events')}"),
          n(f"SELECT count(*) FROM {LIVE}"), "um pedido por pedido com evento na silver")
registrar("gold_orders_live", "granularidade_sem_duplicata",
          n(f"SELECT count(*) FROM {LIVE}"), n(f"SELECT count(DISTINCT order_id) FROM {LIVE}"), "grão: order_id")
registrar("gold_orders_live", "status_igual_silver", 0, n(f'''
SELECT count(*) FROM ({ESPERADO}) e FULL OUTER JOIN {LIVE} l ON e.order_id = l.order_id
WHERE e.order_id IS NULL OR l.order_id IS NULL OR e.st <> l.status OR e.seq <> l.event_sequence'''),
          "pedidos com status ou sequência diferentes (ou só de um lado)")
registrar("gold_orders_live", "status_igual_fact_orders", 0, n(f'''
SELECT count(*) FROM {LIVE} l JOIN {gold('fact_orders')} f ON f.order_id = l.order_id WHERE l.status <> f.status_atual'''),
          "pedidos com status diferente da gold batch")
registrar("gold_orders_live", "entrega_igual_fact_orders", 0, n(f'''
SELECT count(*) FROM {LIVE} l JOIN {gold('fact_orders')} f ON f.order_id = l.order_id
WHERE NOT (l.delivered_at <=> f.order_delivered_customer_date)'''), "pedidos com data de entrega diferente da gold batch")
registrar("gold_orders_live", "etapas_igual_silver", 0, n(f'''
SELECT count(*) FROM ({ESPERADO}) e JOIN {LIVE} l ON e.order_id = l.order_id
WHERE NOT (e.created_at <=> l.created_at AND e.approved_at <=> l.approved_at AND e.shipped_at <=> l.shipped_at AND e.delivered_at <=> l.delivered_at)'''),
          "pedidos com algum instante de etapa diferente")
registrar("gold_orders_live", "estimada_igual_silver", 0, n(f'''
SELECT count(*) FROM {LIVE} l LEFT JOIN {silver('orders')} o ON o.order_id = l.order_id
WHERE NOT (l.estimated_delivery_date <=> o.order_estimated_delivery_date)'''), "pedidos com data estimada diferente da silver")
registrar("gold_orders_live", "pedido_sem_evento_fora_do_live",
          n(f"SELECT count(*) FROM {silver('orders')} o LEFT ANTI JOIN {silver('order_status_events')} e ON o.order_id = e.order_id"),
          n(f"SELECT count(*) FROM {silver('orders')} o LEFT ANTI JOIN {LIVE} l ON o.order_id = l.order_id"),
          "pedidos sem evento (status desconhecido na gold batch) não entram no estado ao vivo")
registrar("order_status_events", "evento_sem_pedido", 0,
          n(f"SELECT count(*) FROM {silver('order_status_events')} e LEFT ANTI JOIN {silver('orders')} o ON e.order_id = o.order_id"),
          "eventos órfãos (risco apontado no desenho)")

## Grupo 2 — micro-lotes: nada perdido, latência e tardios

`ops.streaming_batches` tem um registro por micro-lote da produção (os testes gravam em tabelas de prova). Duas conferências:

- **Nenhum evento perdido:** os eventos distintos `(order_id, event_sequence)` da bronze limpa são exatamente a soma de `events_in` dos micro-lotes. Se o Spark descartasse algum evento (o que o relógio `event_ts` fazia, ADR 0013), a soma seria menor.
- **Latência:** nenhum micro-lote **incremental** (até 10.000 eventos; acima disso é a carga inicial ou uma recuperação) passou de 15 minutos entre a gravação do arquivo na landing e o fim do `MERGE`.

In [ ]:
# O QUE FAZ: registra as verificações do grupo 2 (micro-lotes).
#
# FUNÇÕES NATIVAS (SQL):
# - count(DISTINCT a, b): combinações diferentes. sum(...): soma. max(...): maior valor.
# - count_if(cond): micro-lotes que cumprem a condição.
registrar("streaming_batches", "nenhum_evento_perdido", n(f'''
SELECT count(*) FROM (SELECT DISTINCT order_id, CAST(event_sequence AS INT) FROM {EVENTOS_BRONZE}
                      WHERE _corrupt_record IS NULL AND order_id IS NOT NULL)'''),
          n(f"SELECT sum(events_in) FROM {BATCHES}"), "eventos distintos da bronze = eventos que chegaram ao MERGE")
registrar("streaming_batches", "latencia_dentro_do_sla", 0,
          n(f"SELECT count_if(max_latency_seconds > {SLA_SEGUNDOS}) FROM {BATCHES} WHERE events_in <= {LOTE_GRANDE}"),
          f"micro-lotes incrementais acima de {SLA_SEGUNDOS // 60} min")
registrar("streaming_batches", "latencia_maxima_s", None,
          n(f"SELECT round(max(max_latency_seconds)) FROM {BATCHES} WHERE events_in <= {LOTE_GRANDE}"), "maior latência de um micro-lote incremental, em segundos")
registrar("streaming_batches", "microlotes_incrementais", None, n(f"SELECT count(*) FROM {BATCHES} WHERE events_in <= {LOTE_GRANDE}"), "micro-lotes medidos")
registrar("streaming_batches", "tardios_contados", None, n(f"SELECT sum(late_events) FROM {BATCHES}"), "eventos mais velhos que o maior event_ts gravado menos 4 dias (aplicados)")

## Grupo 3 — risco de atraso

A regra do ADR 0011 é **reescrita** aqui, com outra forma: em vez de um `CASE` com comparação de datas, calcula-se `d = datediff(data estimada, hoje)` e decide-se por `d`. "Hoje" vem do maior `event_ts` da **silver**. O esperado usa o status e a data estimada da silver; o obtido é a coluna `risco_atraso` de `gold_orders_live`. Confere também a view `vw_pedidos_em_risco`.

In [ ]:
# O QUE FAZ: registra as verificações do grupo 3 (risco).
#
# FUNÇÕES NATIVAS (SQL):
# - CAST(max(event_ts) AS DATE): a data simulada corrente. datediff(a, b): dias de a até b.
# - CASE WHEN ... END: escolhe o nível por d (dias até a data estimada).
# - a <=> b: igualdade que trata nulo = nulo como verdadeiro.
# - dias(d): a data como número inteiro (dias desde 1970-01-01), para caber na coluna de esperado e obtido.
hoje = spark.sql(f"SELECT CAST(max(event_ts) AS DATE) FROM {silver('order_status_events')}").first()[0]
hoje_live = spark.sql(f"SELECT CAST(max(event_ts) AS DATE) FROM {LIVE}").first()[0]
from datetime import date


def dias(d):
    """Dias desde 1970-01-01 (a data como número inteiro)."""
    return (d - date(1970, 1, 1)).days


registrar("gold_orders_live", "data_simulada_igual", dias(hoje), dias(hoje_live), f"hoje = {hoje} (maior event_ts)")

abertos = ", ".join(f"'{s}'" for s in ABERTOS)
RISCO_ESPERADO = f'''
  SELECT e.order_id, e.st,
         CASE WHEN e.st NOT IN ({abertos}) THEN NULL
              WHEN o.order_estimated_delivery_date IS NULL THEN 'normal'
              WHEN datediff(CAST(o.order_estimated_delivery_date AS DATE), DATE'{hoje}') < 0 THEN 'atrasado'
              WHEN e.st <> 'shipped' AND datediff(CAST(o.order_estimated_delivery_date AS DATE), DATE'{hoje}') <= {N_EM_RISCO} THEN 'em_risco'
              ELSE 'normal' END AS nivel
  FROM (SELECT order_id, max_by(status, event_sequence) AS st FROM {silver('order_status_events')} GROUP BY order_id) e
  JOIN {silver('orders')} o ON o.order_id = e.order_id
'''
registrar("gold_orders_live", "risco_igual_calculo_independente", 0, n(f'''
SELECT count(*) FROM ({RISCO_ESPERADO}) r FULL OUTER JOIN {LIVE} l ON r.order_id = l.order_id
WHERE r.order_id IS NULL OR l.order_id IS NULL OR NOT (r.nivel <=> l.risco_atraso)'''), "pedidos com nível de risco diferente do recalculado")
registrar("gold_orders_live", "risco_so_pedidos_abertos", 0, n(f'''
SELECT count(*) FROM {LIVE} WHERE (status NOT IN ({abertos}) AND risco_atraso IS NOT NULL) OR (status IN ({abertos}) AND risco_atraso IS NULL)'''),
          "pedido fechado com risco, ou aberto sem risco")

for nivel in ("atrasado", "em_risco", "normal"):
    registrar("gold_orders_live", f"pedidos_{nivel}", None, n(f"SELECT count(*) FROM {LIVE} WHERE risco_atraso = '{nivel}'"), f"nível {nivel} na data simulada {hoje}")

registrar(VIEW_RISCO.split(".")[-1], "lista_igual_atrasados_e_em_risco",
          n(f"SELECT count(*) FROM {LIVE} WHERE risco_atraso IN ('atrasado', 'em_risco')"), n(f"SELECT count(*) FROM {VIEW_RISCO}"), "a view lista exatamente os atrasados e em risco")
registrar(VIEW_RISCO.split(".")[-1], "lista_sem_pedido_fechado", 0,
          n(f"SELECT count(*) FROM {VIEW_RISCO} WHERE status NOT IN ({abertos})"), "pedido fechado na lista")
registrar(VIEW_RISCO.split(".")[-1], "lista_sem_duplicata", n(f"SELECT count(*) FROM {VIEW_RISCO}"), n(f"SELECT count(DISTINCT order_id) FROM {VIEW_RISCO}"), "grão: order_id")

## Grupo 4 — idempotência

O Free Edition não permite executar um notebook dentro de outro, então esta validação **não roda o monitor**: ela lê o resultado da conferência que o próprio notebook `01_gold_orders_live` (`teste_leitura = conferencia`) grava em `ops.streaming_conferencias` e **recalcula o hash do estado aqui, em SQL**. A conferência só vale se foi feita **depois** do último micro-lote gravado e se o hash dela é o do estado de agora; caso contrário a verificação reprova e pede para rodar a conferência de novo.

A conferência faz duas provas: reler **toda a bronze com um checkpoint novo** numa tabela de prova e obter o mesmo estado (replay), e rodar o stream real **sem arquivo novo** e obter o mesmo estado.

In [ ]:
# O QUE FAZ: registra as verificações do grupo 4 (idempotência), lendo o registro da última conferência do monitor.
#
# FUNÇÕES NATIVAS (SQL):
# - pmod(xxhash64(colunas), 1000000007): hash da linha reduzido, para a soma não estourar o bigint (modo ANSI).
# - ORDER BY conferido_em DESC LIMIT 1: a conferência mais recente. max(processed_at): último micro-lote gravado.
# - spark.catalog.tableExists(nome): True se a tabela existe.
# - bool(x) e int(x): verdadeiro/falso e inteiro (None vira falso).
hash_atual = n(f'''
    SELECT sum(pmod(xxhash64(order_id, status, event_sequence, event_ts, created_at, approved_at, shipped_at, delivered_at, estimated_delivery_date), 1000000007))
    FROM {LIVE}''')
linhas_atuais = n(f"SELECT count(*) FROM {LIVE}")
ultimo_lote = spark.sql(f"SELECT max(processed_at) FROM {BATCHES}").first()[0]

tem_tabela = spark.catalog.tableExists(f"{CATALOG}.ops.streaming_conferencias")
conferencia = spark.sql(f"SELECT * FROM {CATALOG}.ops.streaming_conferencias ORDER BY conferido_em DESC LIMIT 1").first() if tem_tabela else None
if conferencia is None:
    print("ATENÇÃO: nenhuma conferência encontrada: rode 01_gold_orders_live com teste_leitura = conferencia")
atual = conferencia is not None and conferencia["conferido_em"] >= ultimo_lote
registrar("gold_orders_live", "conferencia_depois_do_ultimo_lote", 1, int(atual), "a conferência do monitor é mais nova que o último micro-lote")
registrar("gold_orders_live", "conferencia_hash_igual_ao_atual", hash_atual, conferencia["hash_estado"] if conferencia else None,
          f"{linhas_atuais} linhas no estado atual; hash recalculado aqui em SQL")
registrar("gold_orders_live", "replay_checkpoint_novo", 1, int(bool(conferencia and conferencia["replay_ok"])), "reler toda a bronze com checkpoint novo dá o mesmo estado")
registrar("gold_orders_live", "idempotencia_sem_arquivo_novo", 1, int(bool(conferencia and conferencia["sem_arquivo_novo_ok"])), "rodar o stream real sem arquivo novo não muda o estado")

## Gravação e resultado final

Grava todas as linhas em `quality.validacao_live` (tabela de histórico: cada validação acrescenta as suas) e falha o notebook se alguma verificação estiver reprovada, para que um job de orquestração (Fase 7) pare no problema.

In [ ]:
# O QUE FAZ: grava os resultados, mostra o resumo e interrompe com erro se houver verificação reprovada.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(linhas, schema): cria um DataFrame a partir de uma lista de tuplas.
# - DataFrame.withColumns({...}): acrescenta colunas.
# - DataFrame.write.mode("append").saveAsTable(nome): acrescenta as linhas à tabela (o histórico fica).
# - display(df): mostra o DataFrame como tabela interativa.
# - raise RuntimeError(texto): interrompe a execução com erro.
df = (
    spark.createDataFrame(
        resultados,
        "entidade string, verificacao string, esperado long, obtido long, escala long, ok boolean, detalhe string",
    )
    .withColumns({"validacao_id": F.lit(VALIDACAO_ID), "verificado_em": F.current_timestamp()})
    .select("validacao_id", "verificado_em", "entidade", "verificacao", "esperado", "obtido", "escala", "ok", "detalhe")
)
df.write.mode("append").saveAsTable(TABELA_RESULTADOS)

reprovadas = [r for r in resultados if r[5] is False]
total = [r for r in resultados if r[5] is not None]
print(f"\n{len(total) - len(reprovadas)} de {len(total)} verificações OK")
display(df.where(F.col("ok") == False).select("entidade", "verificacao", "esperado", "obtido", "escala", "detalhe"))  # noqa: E712

if reprovadas:
    raise RuntimeError(f"{len(reprovadas)} verificação(ões) reprovada(s): veja a tabela {TABELA_RESULTADOS}.")